# 02 — PySpark: fundamentos

Trabajar con PySpark DataFrames y realizar transformaciones y análisis sobre los datos.

## 1. Crear un DataFrame

Cargar la tabla de clientes como DataFrame.

In [0]:
df_customer = spark.table("samples.tpcds_sf1.customer")

display(df_customer.limit(5))

## 2. Conocer el DataFrame

Revisar la estructura y cantidad de registros.

In [0]:
df_customer.printSchema()

In [0]:
print("Filas:", df_customer.count())
print("Columnas:", len(df_customer.columns))

## 3. Seleccionar columnas

Seleccionar únicamente las columnas necesarias.

In [0]:
df_customer.select(
    "c_customer_sk",
    "c_first_name",
    "c_last_name",
    "c_birth_country"
).show(10)

## 4. Filtrar registros

Filtrar registros según una o varias condiciones.

In [0]:
df_customer.filter(
    df_customer.c_birth_country == "UNITED STATES"
).select(
    "c_customer_sk",
    "c_first_name",
    "c_last_name",
    "c_birth_country"
).show(10)

Combinar varias condiciones.

In [0]:
df_customer.filter(
    (df_customer.c_birth_country == "UNITED STATES") &
    (df_customer.c_birth_year >= 1990)
).select(
    "c_customer_sk",
    "c_first_name",
    "c_last_name",
    "c_birth_year"
).show(10)

## 5. Crear nuevas columnas

Crear columnas calculadas a partir de los datos existentes.

In [0]:
from pyspark.sql import functions as F

df_item = spark.table("samples.tpcds_sf1.item")

df_item = df_item.withColumn(
    "rango_precio",
    F.when(F.col("i_current_price") < 35, "Bajo")
     .when(F.col("i_current_price") < 70, "Medio")
     .otherwise("Alto")
)

display(
    df_item.select(
        "i_item_id",
        "i_category",
        "i_current_price",
        "rango_precio"
    ).limit(10)
)

## 6. Agregaciones

Obtener indicadores por categoría.

In [0]:
df_item.groupBy("i_category").agg(
    F.count("*").alias("cantidad_productos"),
    F.round(F.avg("i_current_price"), 2).alias("precio_promedio"),
    F.round(F.min("i_current_price"), 2).alias("precio_minimo"),
    F.round(F.max("i_current_price"), 2).alias("precio_maximo")
).orderBy(
    F.col("precio_promedio").desc()
).show()

## 7. Trabajar con fechas

Consultar y filtrar información por fecha y año.

In [0]:
df_date = spark.table("samples.tpcds_sf1.date_dim")

df_date.select(
    "d_date",
    "d_year",
    "d_moy",
    "d_qoy"
).filter(F.col("d_year").between(2000, 2002)
).orderBy("d_date").show(30)

## 8. JOIN entre DataFrames

Relacionar ventas con productos para obtener ingresos por categoría.

In [0]:
df_sales = spark.table("samples.tpcds_sf1.store_sales")

df_sales_category = (
    df_sales.alias("ss")
    .join(
        df_item.alias("i"),
        F.col("ss.ss_item_sk") == F.col("i.i_item_sk"),
        "inner"
    )
    .groupBy(
        F.col("i.i_category")
    )
    .agg(
        F.round(
            F.sum("ss.ss_net_paid"), 2
        ).alias("ingresos")
    )
    .orderBy(
        F.col("ingresos").desc()
    )
)

display(df_sales_category)

## 9. JOIN con varias tablas

Relacionar ventas, productos y fechas.

In [0]:
df_sales_year = (
    df_sales.alias("ss")
    .join(
        df_item.alias("i"),
        F.col("ss.ss_item_sk") == F.col("i.i_item_sk"),
        "inner"
    )
    .join(
        df_date.alias("d"),
        F.col("ss.ss_sold_date_sk") == F.col("d.d_date_sk"),
        "inner"
    )
    .groupBy(
        F.col("d.d_year"),
        F.col("i.i_category")
    )
    .agg(
        F.round(
            F.sum("ss.ss_net_paid"), 2
        ).alias("ingresos")
    )
    .orderBy(
        "d_year",
        F.col("ingresos").desc()
    )
)

display(df_sales_year)

## 10. Funciones de ventana

Calcular rankings manteniendo el detalle de los registros.

In [0]:
from pyspark.sql.window import Window

window_year = Window.partitionBy(
    "d_year"
).orderBy(
    F.col("ingresos").desc()
)

df_ranking = df_sales_year.withColumn(
    "ranking_anual",
    F.rank().over(window_year)
)

display(df_ranking)

Filtro para solo mostrar los 3 mejores por años

In [0]:
display(
    df_ranking.filter(
        F.col("ranking_anual") <= 3
    )
)

## 11. Transformaciones encadenadas

Combinar varias transformaciones en un mismo flujo.

In [0]:
df_top_categories = (
    df_sales.alias("ss")
    .join(
        df_item.alias("i"),
        F.col("ss.ss_item_sk") == F.col("i.i_item_sk"),
        "inner"
    )
    .join(
        df_date.alias("d"),
        F.col("ss.ss_sold_date_sk") == F.col("d.d_date_sk"),
        "inner"
    )
    .groupBy(
        "d.d_year",
        "i.i_category"
    )
    .agg(
        F.round(
            F.sum("ss.ss_net_paid"), 2
        ).alias("ingresos")
    )
    .withColumn(
        "ranking_anual",
        F.rank().over(
            Window.partitionBy("d_year")
            .orderBy(F.col("ingresos").desc())
        )
    )
    .filter(
        F.col("ranking_anual") <= 3
    )
    .orderBy(
        "d_year",
        "ranking_anual"
    )
)

display(df_top_categories)

## 12. Reto final

Obtener las tres categorías con mayores ingresos por año.

In [0]:
df_reto = (
    df_sales.alias("ss")
    .join(
        df_item.alias("i"),
        F.col("ss.ss_item_sk") == F.col("i.i_item_sk"),
        "inner"
    )
    .join(
        df_date.alias("d"),
        F.col("ss.ss_sold_date_sk") == F.col("d.d_date_sk"),
        "inner"
    )
    .groupBy(
        "d.d_year",
        "i.i_category"
    )
    .agg(
        F.round(
            F.sum("ss.ss_net_paid"), 2
        ).alias("ingresos")
    )
)

window_year = Window.partitionBy(
    "d_year"
).orderBy(
    F.col("ingresos").desc()
)

df_reto = (
    df_reto
    .withColumn(
        "posicion",
        F.rank().over(window_year)
    )
    .filter(
        F.col("posicion") <= 3
    )
    .orderBy(
        "d_year",
        "posicion"
    )
)

display(df_reto)